# Tamban Model 1 ? leakage-aware retraining

This notebook trains a **single-class whole-fish detector**. It audits the original COCO dataset, retains valid difficult examples, removes only exact duplicates with matching labels, groups near-duplicates and capture bursts, writes a clean YOLO dataset, trains a pretrained YOLO26 model, tunes the counting threshold on validation data, and evaluates one locked internal holdout.

## Run checklist

1. Use Google Colab with a **T4 GPU or better**.
2. Put `Fish Defect Detection.v4i.coco.zip` in MyDrive, or set `DATASET_ZIP_OR_DIR` below.
3. Run all cells from top to bottom. Do not skip the setup or dataset-preparation cells.
4. Review `audit/` before accepting a model.
5. Treat the held-out test as an internal estimate, not independent thesis accuracy. Validate later on genuinely new capture sessions with expert-reviewed labels.

The original dataset is never modified. Each run writes to a timestamped output folder under `Tamban_Model1_Retrain/`.


In [ ]:
# TAMBAN Model 1 — COCO to YOLO26, safe audit, leakage-aware training and evaluation
# Export supported: "Fish Defect Detection.v4i.coco" (directory or .zip)
# Generated for Google Colab. The source archive is NEVER edited.

import os, sys, re, json, time, math, hashlib, zipfile, shutil, random, warnings, importlib
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

# ---------------------- USER CONFIGURATION ----------------------
DATASET_NAME = 'Fish Defect Detection.v4i.coco'
DATASET_ZIP_OR_DIR = None       # Set this to an explicit ZIP/folder path when needed
LOCAL_DATASET_CANDIDATES = [
    Path.home() / 'Downloads' / (DATASET_NAME + '.zip'),
    Path.cwd() / (DATASET_NAME + '.zip'),
]
RESUME_FROM_LAST = None        # optional previous run's .../weights/last.pt; recreates same working dataset path
MODEL_WEIGHTS = 'yolo26s.pt'  # pretrained YOLO26 small; retains previous architecture
IMG_SIZE = 768               # 768 is practical on Colab T4; use 960 for small fish if GPU permits
MAX_EPOCHS = 350             # ceiling, NOT a forced length
EARLY_STOP_PATIENCE = 45     # best validation checkpoint, stop after no meaningful improvement
SEED = 42
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
NEAR_DUP_HAMMING = 4         # conservative pHash distance; linked for splits, NOT discarded
RUN_TRAINING = True
RUN_FINAL_TEST = True
RUN_PREDICTION_PREVIEW = True
REQUIRE_GPU = True             # Training on CPU is disabled to avoid misleadingly slow/incomplete runs
SAVE_EVERY_N_EPOCHS = 10
# Set TRUE only when images have reliable WIN_YYYYMMDD_HH_MM_SS filenames.
# Can prevent same-session leakage; may make a split impossible if one huge session.
GROUP_CAPTURE_BURSTS = False
BURST_SECONDS = 180
# If the training dataset has >1 class (e.g. parts or non-fish objects), fail explicitly.
FISH_CLASS_ALIASES = {'fish', 'fishes', 'tamban', 'dried fish', 'dried_fish'}

# ---------------------- DEPENDENCIES ----------------------------
def ensure_deps():
    """Install missing notebook dependencies without modifying the source dataset."""
    import subprocess
    missing = []
    for module in ('ultralytics', 'imagehash'):
        try:
            if module == 'ultralytics':
                from ultralytics import YOLO
            else:
                importlib.import_module(module)
        except Exception as exc:
            missing.append((module, repr(exc)))
    if missing:
        print('Installing or repairing:', ', '.join(name for name, _ in missing))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics', 'ImageHash'])

ensure_deps()
import cv2, yaml, imagehash, torch, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
try:
    from ultralytics import YOLO
    import ultralytics
except Exception as exc:
    raise RuntimeError(
        'Ultralytics could not be imported after dependency repair. '
        'Restart the VS Code notebook kernel and run this cell again. '
        f'Original error: {type(exc).__name__}: {exc}'
    ) from exc

random.seed(SEED)
np.random.seed(SEED)
IS_COLAB = Path('/content').exists()
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA GPU is required for this training notebook. In Colab choose Runtime > Change runtime type > T4 GPU (or better), then restart and Run all.'
    )
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU')
print('Ultralytics:', ultralytics.__version__, 'Torch:', torch.__version__)

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK = Path('/content/tamban_model1_1900_work' if IS_COLAB else './tamban_model1_1900_work').resolve()
BASE_OUTPUT = Path('/content/drive/MyDrive/Tamban_Model1_Retrain' if IN_COLAB else './Tamban_Model1_Retrain').resolve()
OUT = BASE_OUTPUT / f'model1_{STAMP}'
for d in [WORK, OUT, OUT/'audit', OUT/'figures', OUT/'metrics', OUT/'predictions']:
    d.mkdir(parents=True, exist_ok=True)
(OUT/'environment.json').write_text(json.dumps({
    'date_utc': STAMP, 'python': sys.version, 'ultralytics': ultralytics.__version__,
    'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0) if DEVICE == 0 else 'cpu',
    'model_weights': MODEL_WEIGHTS, 'imgsz': IMG_SIZE, 'seed': SEED,
    'max_epochs': MAX_EPOCHS, 'patience': EARLY_STOP_PATIENCE
}, indent=2), encoding='utf-8')
print('Output directory:', OUT)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def safe_extract(zpath, dest):
    with zipfile.ZipFile(zpath) as z:
        for item in z.infolist():
            target = (dest/item.filename).resolve()
            if not target.is_relative_to(dest.resolve()):
                raise ValueError(f'Unsafe ZIP path: {item.filename}')
            if item.file_size > 3 * 1024**3:
                raise ValueError(f'Unexpectedly huge ZIP entry: {item.filename}')
        z.extractall(dest)

def locate_dataset():
    if DATASET_ZIP_OR_DIR:
        p = Path(DATASET_ZIP_OR_DIR).expanduser()
        if not p.exists():
            raise FileNotFoundError(f'Configured dataset path not found: {p}')
        return p
    if IN_COLAB:
        roots = [Path('/content'), Path('/content/drive/MyDrive')]
    else:
        roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.home() / 'Downloads']
        roots.extend(p.parent for p in LOCAL_DATASET_CANDIDATES)
    seen = set()
    for root in roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        for name in (DATASET_NAME + '.zip', DATASET_NAME, DATASET_NAME + '.ZIP'):
            p = root / name
            if p.exists():
                return p
    if IN_COLAB:
        print('No dataset found in Colab root or MyDrive root.')
        print('Best for large ZIPs: put the ZIP in Google Drive, then set DATASET_ZIP_OR_DIR above.')
        print('Otherwise select your dataset ZIP in the upload window.')
        selected = files.upload()
        zips = [Path('/content')/n for n in selected if Path(n).suffix.lower() == '.zip']
        if len(zips) != 1:
            raise RuntimeError('Upload exactly one COCO dataset ZIP, or set DATASET_ZIP_OR_DIR.')
        return zips[0]
    raise FileNotFoundError(
        f'Could not find {DATASET_NAME}.zip. Put it in {Path.home() / 'Downloads'} or set DATASET_ZIP_OR_DIR to its full path.'
    )

SOURCE = locate_dataset()
print('Dataset source:', SOURCE)
if SOURCE.is_file():
    EXTRACT = WORK/'extracted_coco'
    if EXTRACT.exists():
        shutil.rmtree(EXTRACT)
    EXTRACT.mkdir(parents=True)
    safe_extract(SOURCE, EXTRACT)
else:
    EXTRACT = SOURCE


## Dataset preparation and leakage control

This consolidated cell reloads the original COCO JSON files from disk. It validates image dimensions and boxes, records quarantined records, preserves reviewed whole-fish labels, groups related images, creates group-exclusive train/validation/test splits, and writes YOLO labels.

The default keeps natural overlapping fish because overlap is a valid detection case. Set `EXCLUDE_REVIEWED_OVERLAP_IMAGES = True` only when the CSV contains manually reviewed false annotations.


In [ ]:
# ================================================================
# TAMBAN MODEL 1 — COMPLETE COCO -> YOLO DATA PREPARATION
# For: Fish Defect Detection.v4i.coco (fish + reject = Fish 0)
# Run in ONE Colab cell AFTER original ZIP extraction / Drive mount.
# Requires EXTRACT, WORK, OUT from notebook setup. Does NOT rely on
# old COCO_FILES, records, split assignments or category mapping.
# ================================================================
import json, math, hashlib, random, re, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict, Counter
import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Set False to KEEP genuinely overlapping fish (recommended for robustness).
# True excludes only images listed in your PREVIOUSLY REVIEWED overlap CSV.
EXCLUDE_REVIEWED_OVERLAP_IMAGES = False
NEAR_DUP_HAMMING = 4
GROUP_CAPTURE_BURSTS = True
BURST_SECONDS = 15
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SEED = 42
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Make this cell safe to rerun independently after a kernel restart.
if not all(var in globals() for var in ('EXTRACT', 'WORK', 'OUT')):
    DATASET_NAME = globals().get('DATASET_NAME', 'Fish Defect Detection.v4i.coco')
    DATASET_ZIP_OR_DIR = globals().get('DATASET_ZIP_OR_DIR')
    IS_COLAB = Path('/content').exists()
    IN_COLAB = globals().get('IN_COLAB', False)
    if DATASET_ZIP_OR_DIR:
        source = Path(DATASET_ZIP_OR_DIR).expanduser()
    else:
        roots = [Path('/content'), Path('/content/drive/MyDrive')] if IN_COLAB else [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.home() / 'Downloads']
        source = next((candidate for root in roots for candidate in (root / (DATASET_NAME + '.zip'), root / DATASET_NAME, root / (DATASET_NAME + '.ZIP')) if candidate.exists()), None)
    if source is None or not source.exists():
        raise FileNotFoundError(
            f'Could not find {DATASET_NAME}.zip. Put it in {Path.home() / "Downloads"} or set DATASET_ZIP_OR_DIR to its full path.'
        )
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') if 'datetime' in globals() else 'local'
    WORK = (Path('/content/tamban_model1_1900_work') if IS_COLAB else Path.cwd() / 'tamban_model1_1900_work').resolve()
    OUT = ((Path('/content/drive/MyDrive/Tamban_Model1_Retrain') if IN_COLAB else Path.cwd() / 'Tamban_Model1_Retrain') / f'model1_{stamp}').resolve()
    WORK.mkdir(parents=True, exist_ok=True)
    for folder in (OUT, OUT / 'audit', OUT / 'figures'):
        folder.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        EXTRACT = WORK / 'extracted_coco'
        if EXTRACT.exists():
            shutil.rmtree(EXTRACT)
        EXTRACT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as archive:
            archive.extractall(EXTRACT)
    else:
        EXTRACT = source
EXTRACT, WORK, OUT = map(lambda p: Path(p).resolve(), (EXTRACT, WORK, OUT))
if not EXTRACT.is_dir():
    raise FileNotFoundError(f'Original extracted COCO folder missing: {EXTRACT}')
if OUT == EXTRACT or EXTRACT in OUT.parents:
    raise RuntimeError('OUT must not be inside the original COCO extraction folder.')
AUDIT, FIGS = OUT/'audit', OUT/'figures'
AUDIT.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

# 1) Load ORIGINAL JSON files from disk, never reuse previously filtered COCO_FILES.
COCO_FILES = []
for p in sorted(EXTRACT.rglob('*.json')):
    if p.stat().st_size > 250*1024*1024: continue
    try:
        j = json.loads(p.read_text(encoding='utf-8-sig'))
    except (ValueError, UnicodeError):
        continue
    if isinstance(j, dict) and all(k in j for k in ('images','annotations','categories')):
        COCO_FILES.append((p,j))
if not COCO_FILES:
    raise RuntimeError('No original COCO JSON files found under EXTRACT.')
original_n = sum(len(c['images']) for _,c in COCO_FILES)
print('Original COCO files:', [(str(p.relative_to(EXTRACT)),len(c['images'])) for p,c in COCO_FILES])
print('Original image entries:', original_n)

# 2) Preload reviewed overlap report (DO NOT mark every natural overlap as an error).
def norm_split(v):
    v = str(v).strip().lower()
    return {'valid':'val','validation':'val','training':'train','testing':'test'}.get(v,v)
def source_split(p):
    for part in p.relative_to(EXTRACT).parts:
        s = norm_split(part)
        if s in ('train','val','test'): return s
    return 'unspecified'

overlap_path = AUDIT/'potential_duplicate_fish_boxes.csv'
flagged_images = set()
if EXCLUDE_REVIEWED_OVERLAP_IMAGES:
    if not overlap_path.exists():
        raise FileNotFoundError(
            f'Reviewed overlap CSV not found: {overlap_path}. '
            'Restore the file or set EXCLUDE_REVIEWED_OVERLAP_IMAGES=False.'
        )
    flagged_df = pd.read_csv(overlap_path)
    if not {'split','image'}.issubset(flagged_df.columns):
        raise RuntimeError('Overlap CSV must contain split and image columns.')
    flagged_images = {(norm_split(x),Path(str(y)).name)
                      for x,y in zip(flagged_df['split'],flagged_df['image'])}
    print('Reviewed overlapping-image keys to exclude:',len(flagged_images))

# 3) Build name index for safe image resolution.
all_paths = [p for p in EXTRACT.rglob('*')
             if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
by_basename = defaultdict(list)
for p in all_paths: by_basename[p.name].append(p)
def find_image(jpath, filename):
    filename = str(filename).replace('\\','/').lstrip('/')
    name = Path(filename).name
    candidates = [jpath.parent/filename, EXTRACT/filename,
                  jpath.parent/name, jpath.parent/'images'/name]
    for p in candidates:
        if p.is_file(): return p.resolve()
    local = [p for p in by_basename[name] if p.parent == jpath.parent]
    if len(local)==1: return local[0].resolve()
    if len(by_basename[name])==1: return by_basename[name][0].resolve()
    return None

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def phash(image):
    small = cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),
                       (32,32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(small)[:8,:8]
    med = np.median(low.ravel()[1:])
    ans=0
    for flag in (low > med).ravel(): ans=(ans<<1)|int(flag)
    return ans

records, issues, fixes, exclusions = [], [], [], []
category_counts, category_map = [], {}
for jpath,coco in COCO_FILES:
    split = source_split(jpath)
    ids = {int(c['id']):str(c['name']).strip().lower() for c in coco['categories']}
    keep_ids = {i for i,name in ids.items() if name in ('fish','reject')}
    reject_ids = {i for i,name in ids.items() if name=='reject'}
    category_map[str(jpath.relative_to(EXTRACT))] = {str(i):name for i,name in ids.items()}
    if not keep_ids:
        raise RuntimeError(f'No fish/reject classes in {jpath}')
    ann_by_image = defaultdict(list)
    for a in coco['annotations']: ann_by_image[a['image_id']].append(a)
    kept=0; fish_boxes=0; reject_used=0
    for obj in coco['images']:
        key=(split,Path(str(obj.get('file_name',''))).name)
        ref=f"{jpath.relative_to(EXTRACT)}::{obj['id']}::{obj.get('file_name','')}"
        anns=ann_by_image.get(obj['id'],[])
        chosen=[a for a in anns if int(a.get('category_id',-1)) in keep_ids]
        has_reject=any(int(a.get('category_id',-1)) in reject_ids for a in chosen)
        if not chosen:
            issues.append({'record':ref,'reason':'no_whole_fish_boxes','detail':''}); continue
        if key in flagged_images:
            exclusions.append({'record':ref,'reason':'reviewed_true_overlap','fish_boxes':len(chosen)})
            continue
        img_path = find_image(jpath,obj.get('file_name',''))
        if img_path is None:
            issues.append({'record':ref,'reason':'image_missing_or_ambiguous','detail':''});continue
        image=cv2.imread(str(img_path))
        if image is None:
            issues.append({'record':ref,'reason':'corrupt_image','detail':str(img_path)});continue
        H,W=image.shape[:2]
        if min(H,W)<32:
            issues.append({'record':ref,'reason':'too_small','detail':f'{W}x{H}'});continue
        try: dim_ok=abs(W-int(obj.get('width',W)))<=2 and abs(H-int(obj.get('height',H)))<=2
        except (ValueError,TypeError): dim_ok=False
        if not dim_ok:
            issues.append({'record':ref,'reason':'dimension_mismatch','detail':f'actual {W}x{H}'});continue
        boxes=[]; bad=None; dup=0
        for a in chosen:
            if a.get('iscrowd',0): bad='iscrowd';break
            try: x,y,bw,bh=map(float,a['bbox'])
            except (KeyError,TypeError,ValueError): bad='bad_bbox_format';break
            if not all(math.isfinite(v) for v in (x,y,bw,bh)) or bw<=0 or bh<=0:
                bad='invalid_bbox';break
            exceed=max(0.,-x,-y,x+bw-W,y+bh-H)
            if exceed>2: bad='out_of_bounds_gt_2px';break
            x1,y1,x2,y2=max(0.,x),max(0.,y),min(float(W),x+bw),min(float(H),y+bh)
            if x2-x1<2 or y2-y1<2: bad='bbox_under_2px';break
            box=[(x1+x2)/(2*W),(y1+y2)/(2*H),(x2-x1)/W,(y2-y1)/H]
            if any(all(abs(v-u)<1e-5 for v,u in zip(box,old)) for old in boxes):
                dup+=1;continue
            boxes.append(box)
            if exceed>0: fixes.append({'record':ref,'action':'clip_bbox_le_2px','annotation':a.get('id')})
        if bad or not boxes:
            issues.append({'record':ref,'reason':bad or 'no_boxes_after_checks','detail':''});continue
        if dup: fixes.append({'record':ref,'action':'remove_identical_boxes','count':dup})
        boxes.sort()
        small=cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),(128,128),interpolation=cv2.INTER_AREA)
        records.append({'reference':ref,'source_path':img_path,'source_split':split,
                        'width':W,'height':H,'boxes':boxes,'fish_count':len(boxes),
                        'contains_reject':has_reject,'brightness':float(small.mean()),
                        'blur_metric':float(cv2.Laplacian(small,cv2.CV_64F).var()),
                        'sha':sha(img_path),'phash':phash(image),'duplicate_annotations_removed':dup})
        kept+=1;fish_boxes+=len(boxes);reject_used+=int(has_reject)
    category_counts.append({'split':split,'original_images':len(coco['images']),
                            'usable_before_exact_dedup':kept,'whole_fish_boxes':fish_boxes,
                            'images_with_reject_retained':reject_used})

pd.DataFrame(category_counts).to_csv(AUDIT/'category_counts.csv',index=False)
pd.DataFrame(issues,columns=['record','reason','detail']).to_csv(AUDIT/'quarantined_records.csv',index=False)
pd.DataFrame(fixes).to_csv(AUDIT/'automatic_label_fixes.csv',index=False)
pd.DataFrame(exclusions).to_csv(AUDIT/'excluded_reviewed_overlap_images.csv',index=False)
print('\nAfter label checks:',len(records),'images;',sum(r['fish_count'] for r in records),'whole-fish boxes')
print('Images containing rejected fish BEFORE exact dedup:',sum(r['contains_reject'] for r in records))
print('Excluded reviewed overlap images:',len(exclusions),' other issues:',len(issues))
if len(records)<100: raise RuntimeError('Too few usable images; inspect audit reports.')
if sum(r['contains_reject'] for r in records)==0:
    raise RuntimeError('Zero recovered Reject images. Verify original COCO files/categories.')

# 4) Remove byte-identical images ONLY when all labels agree.
sha_groups=defaultdict(list)
for r in records: sha_groups[r['sha']].append(r)
unique,dropped,conflicts=[],[],[]
for hashval,grp in sha_groups.items():
    sig={tuple(tuple(round(v,5) for v in box) for box in r['boxes']) for r in grp}
    if len(sig)>1:
        conflicts.append({'sha':hashval,'records':[r['reference'] for r in grp]});continue
    grp.sort(key=lambda r:({'test':0,'val':1,'train':2}.get(r['source_split'],3),r['reference']))
    unique.append(grp[0]);dropped+=grp[1:]
(AUDIT/'exact_label_conflicts.json').write_text(json.dumps(conflicts,indent=2))
if conflicts:
    raise RuntimeError(f'{len(conflicts)} exact-image label conflicts. See exact_label_conflicts.json')
records=unique
pd.DataFrame([{'record':r['reference'],'sha':r['sha']} for r in dropped]).to_csv(
    AUDIT/'removed_exact_duplicates.csv',index=False)

# 5) Group Roboflow derivatives, near duplicates, optional capture bursts.
class DSU:
    def __init__(self,n): self.p=list(range(n))
    def root(self,i):
        while self.p[i]!=i:
            self.p[i]=self.p[self.p[i]];i=self.p[i]
        return i
    def union(self,i,j):
        a,b=self.root(i),self.root(j)
        if a!=b:self.p[b]=a

def origin(stem):
    stem=re.sub(r'(?i)_(jpg|jpeg|png|bmp|webp)\.rf\.[a-z0-9]+$','',stem)
    return re.sub(r'(?i)\.rf\.[a-z0-9]+$','',stem).lower()

def burst(stem):
    match=re.search(r'WIN_(\d{8})_(\d\d)_(\d\d)_(\d\d)',stem,re.I)
    if not match:return None
    day,h,m,s=match.groups()
    return (day,(int(h)*3600+int(m)*60+int(s))//BURST_SECONDS)

N=len(records);uf=DSU(N)
origins=defaultdict(list)
for i,r in enumerate(records):origins[origin(r['source_path'].stem)].append(i)
for indexes in origins.values():
    for i in indexes[1:]:uf.union(indexes[0],i)
near_pairs=[]
for i in range(N):
    a=records[i]
    for j in range(i+1,N):
        b=records[j]
        if a['fish_count']!=b['fish_count']:continue
        aspect_a=a['width']/a['height'];aspect_b=b['width']/b['height']
        if abs(aspect_a-aspect_b)/max(aspect_a,aspect_b)>0.035:continue
        if abs(a['brightness']-b['brightness'])>20:continue
        dist=(a['phash']^b['phash']).bit_count()
        if dist<=NEAR_DUP_HAMMING:
            uf.union(i,j)
            if len(near_pairs)<20000:
                near_pairs.append({'a':a['reference'],'b':b['reference'],'phash_distance':dist})
if GROUP_CAPTURE_BURSTS:
    bursts=defaultdict(list)
    for i,r in enumerate(records):
        b=burst(r['source_path'].stem)
        if b is not None:bursts[b].append(i)
    if max((len(v) for v in bursts.values()),default=0)>0.35*N:
        print('WARNING: skipped burst grouping; oversized capture burst')
    else:
        for indexes in bursts.values():
            for i in indexes[1:]:uf.union(indexes[0],i)
group_members=defaultdict(list)
for i,r in enumerate(records):
    r['group']=uf.root(i);group_members[r['group']].append(i)
pd.DataFrame(near_pairs,columns=['a','b','phash_distance']).to_csv(AUDIT/'near_duplicate_pairs.csv',index=False)
print('Unique images:',N,'duplicate groups:',len(group_members),'exact copies removed:',len(dropped))
if len(group_members)<12:
    raise RuntimeError('Too few independent groups; inspect capture-session grouping.')

# 6) Group-exclusive balanced INTERNAL split.
items=list(group_members.items())
rng=random.Random(SEED);rng.shuffle(items);items.sort(key=lambda x:len(x[1]),reverse=True)
target={'train':N*(1-VAL_FRACTION-TEST_FRACTION),'val':N*VAL_FRACTION,'test':N*TEST_FRACTION}
counts={'train':0,'val':0,'test':0};assignments={}
for gid,members in items:
    size=len(members)
    allowed=[s for s in ('train','val','test') if s=='train' or counts[s]+size<=target[s]*1.10]
    if not allowed:allowed=['train']
    def score(s):
        proposed=dict(counts);proposed[s]+=size
        return sum((proposed[k]-target[k])**2/max(target[k],1) for k in counts)
    split=min(allowed,key=score)
    assignments[gid]=split;counts[split]+=size
for r in records:r['split']=assignments[r['group']]
if min(counts.values())<min(30,max(1,int(N*.05))):
    raise RuntimeError(f'Holdout split too small: {counts}; review grouping')
assert all(len({records[i]['split'] for i in inds})==1 for inds in group_members.values())
print('Group-exclusive split:',counts)

# 7) Write clean YOLO dataset, leaving original extracted dataset untouched.
YOLO_DATA=WORK/'yolo_cleaned'
if EXTRACT==YOLO_DATA or EXTRACT in YOLO_DATA.parents:
    raise RuntimeError('YOLO_DATA must not contain original EXTRACT folder')
if any(YOLO_DATA in r['source_path'].parents for r in records):
    raise RuntimeError('Refusing to delete output containing source images.')
if YOLO_DATA.exists():shutil.rmtree(YOLO_DATA)
for split in ('train','val','test'):
    (YOLO_DATA/'images'/split).mkdir(parents=True,exist_ok=True)
    (YOLO_DATA/'labels'/split).mkdir(parents=True,exist_ok=True)
manifest=[]
for i,r in enumerate(records):
    ext=r['source_path'].suffix.lower();base=f'fish_{i:06d}{ext}'
    dst=YOLO_DATA/'images'/r['split']/base
    shutil.copy2(r['source_path'],dst)
    label=YOLO_DATA/'labels'/r['split']/(Path(base).stem+'.txt')
    label.write_text('\n'.join('0 '+' '.join(f'{v:.8f}' for v in b) for b in r['boxes'])+'\n',encoding='utf-8')
    r['dest_path']=dst
    manifest.append({'record':r['reference'],'source':str(r['source_path']),
        'clean_image':str(dst),'original_split':r['source_split'],'split':r['split'],
        'group':r['group'],'fish_boxes':len(r['boxes']),
        'contains_reject':r['contains_reject'],'brightness':r['brightness'],
        'blur_metric':r['blur_metric'],'sha256':r['sha']})
DATA_YAML=YOLO_DATA/'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(YOLO_DATA),'train':'images/train',
    'val':'images/val','test':'images/test','nc':1,'names':{0:'Fish'}},sort_keys=False))
pd.DataFrame(manifest).to_csv(AUDIT/'image_manifest.csv',index=False)

# 8) Persist audit and plots.
recovered=sum(r['contains_reject'] for r in records)
summary={'original_image_entries':original_n,'cleaned_images':N,
    'whole_fish_boxes':sum(len(r['boxes']) for r in records),
    'recovered_images_containing_reject':recovered,
    'exact_duplicates_removed':len(dropped),
    'reviewed_overlap_images_excluded':len(exclusions),
    'quarantined':len(issues),'near_duplicate_pairs':len(near_pairs),
    'distinct_groups':len(group_members),'split_counts':counts,
    'source_categories':category_map,'target_class':'Fish',
    'note':'Grouped internal holdout. NOT an independent capture-session test.'}
(AUDIT/'audit_summary.json').write_text(json.dumps(summary,indent=2))
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].bar(list(counts),list(counts.values()));axs[0].set_title('Grouped image split')
axs[1].hist([len(r['boxes']) for r in records],bins=30)
axs[1].set_title('Fish annotations per image');axs[1].set_xlabel('Fish count')
plt.tight_layout();plt.savefig(FIGS/'dataset_summary.png',dpi=170);plt.show();plt.close()
plt.figure(figsize=(8,4))
for s in ('train','val','test'):
    vals=[r['brightness'] for r in records if r['split']==s]
    plt.hist(vals,bins=20,alpha=.45,label=f'{s} ({len(vals)})')
plt.title('Image brightness by split');plt.xlabel('Brightness (0-255)');plt.legend()
plt.tight_layout();plt.savefig(FIGS/'lighting_distribution.png',dpi=170);plt.show();plt.close()
fig,axs=plt.subplots(3,3,figsize=(12,12))
for row,s in enumerate(('train','val','test')):
    pool=[r for r in records if r['split']==s]
    for col,r in enumerate(random.Random(SEED+row).sample(pool,min(3,len(pool)))):
        im=cv2.imread(str(r['source_path']));im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
        h,w=im.shape[:2];ax=axs[row,col];ax.imshow(im)
        for x,y,bw,bh in r['boxes']:
            ax.add_patch(Rectangle(((x-bw/2)*w,(y-bh/2)*h),bw*w,bh*h,
                                   fill=False,edgecolor='lime',linewidth=1.2))
        ax.set_title(f'{s}: {len(r["boxes"])} fish; Reject={r["contains_reject"]}');ax.axis('off')
for ax in axs.flat:
    if not ax.has_data():ax.axis('off')
plt.tight_layout();plt.savefig(FIGS/'annotation_samples.png',dpi=140);plt.show();plt.close()
for s in ('train','val','test'):
    n_img=len(list((YOLO_DATA/'images'/s).iterdir()))
    n_lab=len(list((YOLO_DATA/'labels'/s).glob('*.txt')))
    assert n_img==n_lab==counts[s],f'{s} output count mismatch'
print('\n'+'='*64)
print('CLEAN MODEL 1 DATASET READY')
print('Original COCO images:',original_n)
print('Clean images:',N,' | Whole fish boxes:',summary['whole_fish_boxes'])
print('Recovered Reject-containing images:',recovered)
print('Reviewed overlapping images excluded:',len(exclusions))
print('Train/Val/Test:',counts)
print('YOLO data.yaml:',DATA_YAML)
print('Audit & graphs:',AUDIT,FIGS)
print('IMPORTANT: GPU must be enabled before YOLO training.')
print('='*64)


## Train the detector

Training uses pretrained YOLO26s with early stopping. Augmentations are intentionally conservative for fish imagery: horizontal flips and moderate rotation/scale are enabled, while vertical flips and mixup are disabled to avoid creating implausible scenes.


In [ ]:
# ---------------------- TRAIN UNTIL VALIDATION PLATEAUS ---------
MAX_DET=max(40,max(r['fish_count'] for r in records)+15)
RUN_DIR=OUT/'runs'/'model1_yolo26s'
if RUN_TRAINING:
    print('\nTraining with early stopping: maximum epochs',MAX_EPOCHS,'patience',EARLY_STOP_PATIENCE)
    print('A mathematical guarantee of "maximum potential" is impossible; select best validation checkpoint.')
    model=YOLO(str(RESUME_FROM_LAST) if RESUME_FROM_LAST else MODEL_WEIGHTS)
    if RESUME_FROM_LAST:
        print('Resuming saved training state from:', RESUME_FROM_LAST)
        model.train(resume=True)
    else:
        model.train(
        data=str(DATA_YAML),epochs=MAX_EPOCHS,patience=EARLY_STOP_PATIENCE,
        imgsz=IMG_SIZE,batch=-1 if DEVICE==0 else 4,device=DEVICE,
        workers=2,cache='disk',project=str(OUT/'runs'),name='model1_yolo26s',exist_ok=True,
        pretrained=True,optimizer='AdamW',lr0=0.001,lrf=0.01,cos_lr=True,
        weight_decay=0.0005,seed=SEED,deterministic=True,amp=DEVICE==0,
        degrees=10,translate=0.06,scale=0.25,shear=0,perspective=0,
        fliplr=0.5,flipud=0.0,hsv_h=0.01,hsv_s=0.25,hsv_v=0.25,
        mosaic=0.35,mixup=0.0,close_mosaic=20,
        plots=True,save=True,save_period=SAVE_EVERY_N_EPOCHS,max_det=MAX_DET,
        val=True,verbose=True,
        )
    RUN_DIR=Path(model.trainer.save_dir)
else:
    print('TRAINING DISABLED — data audit and figures only.')

BEST=RUN_DIR/'weights'/'best.pt'
if RUN_TRAINING and not BEST.exists():
    raise RuntimeError('Best checkpoint not saved — training incomplete; inspect Colab error logs.')


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import json

BASE = Path(
    r"C:\Users\User\Downloads\Tamban_Thesis_Main_Repository"
    r"\Thesis\sardinella-lemuru-ml\notebooks"
)

RUN = (
    BASE / "Tamban_Model1_Retrain"
    / "model1_20261101T161439Z"
    / "runs" / "model1_yolo26s"
)

DATA = (
    BASE / "tamban_model1_1900_work"
    / "yolo_cleaned" / "data.yaml"
)

BEST = RUN / "weights" / "best.pt"

def main():
    assert BEST.exists(), f"Checkpoint missing: {BEST}"
    assert DATA.exists(), f"Dataset YAML missing: {DATA}"

    model = YOLO(str(BEST))

    metrics = model.val(
        data=str(DATA),
        split="test",
        imgsz=768,
        batch=4,
        device=0,
        plots=True,
        project=str(RUN),
        name="final_test_evaluation",
        exist_ok=True
    )

    results = {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map)
    }

    print("\nFINAL MODEL 1 TEST RESULTS")
    print("=" * 40)

    for key, value in results.items():
        print(f"{key}: {value:.4f} ({value*100:.2f}%)")

    output = RUN / "final_test_metrics.json"
    output.write_text(json.dumps(results, indent=2))

    print("\nSaved:", output)

if __name__ == "__main__":
    main()


## Evaluation and deployment

Validation is used for model selection and for tuning the confidence threshold used by fish counting. The internal test split is evaluated only after those choices are locked.

Reported outputs include training curves, precision, recall, mAP@0.5, mAP@0.5:0.95, count MAE/RMSE, exact-count accuracy, per-image test results, prediction previews, and a deployment policy JSON.

> These are grouped internal metrics. They must not be reported as independent real-world generalization accuracy without a new-session expert-labeled test set.


In [ ]:

# ================================================================
# TAMBAN MODEL 1 - MEMORY-SAFE POST-TRAINING EVALUATION
# Paste this complete cell AFTER training, or run with variables
# OUT, BEST, DATA_YAML, IMG_SIZE, DEVICE already defined.
# Does NOT train or modify the saved checkpoint.
# ================================================================

import gc
import json
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

# --------------------------- SETTINGS ---------------------------
# If restarting your notebook/kernel, define OUT, BEST and DATA_YAML
# to their ACTUAL existing paths before running this cell.
if "OUT" not in globals():
    raise RuntimeError("OUT is undefined. Set OUT to your existing run output folder.")
OUT = Path(OUT)

if "BEST" not in globals():
    if "RUN_DIR" in globals():
        BEST = Path(RUN_DIR) / "weights" / "best.pt"
    else:
        raise RuntimeError("BEST is undefined. Set BEST to the trained best.pt path.")
BEST = Path(BEST)

if "DATA_YAML" not in globals():
    if "WORK" in globals():
        DATA_YAML = Path(WORK) / "yolo_cleaned" / "data.yaml"
    else:
        raise RuntimeError("DATA_YAML is undefined. Set it to your cleaned data.yaml path.")
DATA_YAML = Path(DATA_YAML)

IMG_SIZE = int(globals().get("IMG_SIZE", 768))
MAX_DET = int(globals().get("MAX_DET", 132))
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_FINAL_TEST = bool(globals().get("RUN_FINAL_TEST", True))
RUN_PREDICTION_PREVIEW = bool(globals().get("RUN_PREDICTION_PREVIEW", True))
COUNT_IOU = 0.70
LOW_CONF = 0.01
CONF_GRID = np.round(np.arange(0.05, 0.801, 0.05), 2)

if not BEST.is_file():
    raise FileNotFoundError(f"Trained checkpoint not found: {BEST}")
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Clean dataset YAML not found: {DATA_YAML}")

for subdir in ("metrics", "figures", "evaluation", "predictions"):
    (OUT / subdir).mkdir(parents=True, exist_ok=True)

# ---------------------- ON-DISK DATASET INVENTORY ---------------
# Read CURRENT files, not stale in-memory 'records'. This also avoids
# the earlier 291-versus-295 split-count mismatch.
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(data_cfg.get("path") or DATA_YAML.parent)
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def split_inventory(split):
    location = Path(data_cfg[split])
    image_dir = location if location.is_absolute() else dataset_root / location
    if not image_dir.is_dir():
        raise FileNotFoundError(f"{split} images folder missing: {image_dir}")
    label_dir = dataset_root / "labels" / split
    if not label_dir.is_dir():
        raise FileNotFoundError(f"{split} labels folder missing: {label_dir}")
    found = []
    for im in sorted(image_dir.iterdir()):
        if not im.is_file() or im.suffix.lower() not in IMAGE_EXTS:
            continue
        label_file = label_dir / (im.stem + ".txt")
        if not label_file.is_file():
            raise FileNotFoundError(f"Missing label for {im.name}: {label_file}")
        label_lines = [line.strip() for line in label_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        for line in label_lines:
            parts = line.split()
            if len(parts) != 5 or parts[0] != "0":
                raise ValueError(f"Unexpected YOLO label in {label_file}: {line}")
        found.append({"image": im, "gt": len(label_lines)})
    if not found:
        raise RuntimeError(f"No {split} images found in {image_dir}")
    return found

val_rec = split_inventory("val")
test_rec = split_inventory("test") if RUN_FINAL_TEST else []
print(f"Checkpoint: {BEST}")
print(f"Dataset: {DATA_YAML}")
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")
print(f"Validation images: {len(val_rec)} | Test images: {len(test_rec)}")

# Release a finished training wrapper, if present. Its trainer can keep
# optimizer tensors allocated on the GPU even after model.train returns.
training_model = globals().get("model", None)
if isinstance(training_model, YOLO) and getattr(training_model, "trainer", None) is not None:
    globals().pop("model", None)
del training_model
for old in ("val_preds", "test_preds", "previews", "val_metrics", "test_metrics", "best_model"):
    globals().pop(old, None)

def tidy():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tidy()
best_model = YOLO(str(BEST))

# ---------------------- CURVES + EVALUATION ----------------------
def summarize_metrics(m):
    return {
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
        "mAP50": float(m.box.map50),
        "mAP50_95": float(m.box.map),
    }


def plot_training_history(results_path):
    results_path = Path(results_path)
    if not results_path.is_file():
        print("Training history unavailable:", results_path)
        return {"note": "results.csv not found; checkpoint evaluation is still possible"}

    df = pd.read_csv(results_path)
    df.columns = [c.strip() for c in df.columns]
    df.to_csv(OUT / "metrics" / "epoch_history.csv", index=False)
    epochs = pd.to_numeric(df["epoch"], errors="coerce") + 1
    panels = [
        ("Detection metrics", ["metrics/precision(B)", "metrics/recall(B)"]),
        ("mAP by epoch", ["metrics/mAP50(B)", "metrics/mAP50-95(B)"]),
        ("Training losses", [c for c in df.columns if c.startswith("train/") and c.endswith("_loss")]),
        ("Validation losses", [c for c in df.columns if c.startswith("val/") and c.endswith("_loss")]),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, (title, cols) in zip(axes.flat, panels):
        for col in cols:
            if col in df:
                ax.plot(epochs, pd.to_numeric(df[col], errors="coerce"), label=col.split("/")[-1])
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.25)
        if ax.lines:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "training_dashboard.png", dpi=190)
    plt.show()
    plt.close(fig)

    result = {"epochs_completed": len(df)}
    col = "metrics/mAP50-95(B)"
    if col in df:
        vals = pd.to_numeric(df[col], errors="coerce")
        if vals.notna().any():
            best_i = vals.idxmax()
            result.update({
                "best_epoch_by_val_mAP50_95": int(epochs.loc[best_i]),
                "best_val_mAP50_95": float(vals.loc[best_i]),
            })
    return result

RUN_DIR = Path(globals().get("RUN_DIR", BEST.parent.parent))
training_summary = plot_training_history(RUN_DIR / "results.csv")
print("TRAINING SUMMARY:", training_summary)

# Use batch=1 with 4GB cards; no training happens here.
print("\nVALIDATION DETECTION mAP (batch=1)...")
val_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMG_SIZE,
    device=DEVICE, batch=1, workers=0,
    conf=0.001, max_det=MAX_DET, plots=True,
    project=str(OUT / "evaluation"), name="validation_memory_safe",
    exist_ok=True, verbose=False,
)
val_detection = summarize_metrics(val_metrics)
print("INTERNAL VALIDATION METRICS:", val_detection)
del val_metrics
tidy()

# ------------------- MEMORY-SAFE COUNT PREDICTIONS --------------
def predict_cpu_confidences(entries, confidence, save_previews=False):
    """Run exactly one image per call; retain ONLY small CPU arrays."""
    cpu_confidences = []
    nms_saturated = 0
    for idx, item in enumerate(entries, start=1):
        scores = np.empty(0, dtype=np.float32)
        # stream=True prevents accumulation of Ultralytics Results objects.
        with torch.inference_mode():
            for result in best_model.predict(
                source=str(item["image"]),
                conf=float(confidence), iou=COUNT_IOU,
                imgsz=IMG_SIZE, max_det=MAX_DET,
                device=DEVICE, stream=True,
                save=False, verbose=False,
            ):
                if result.boxes is not None and len(result.boxes):
                    scores = result.boxes.conf.detach().float().cpu().numpy().copy()
                if save_previews and idx <= 18:
                    preview_file = OUT / "predictions" / f"test_{idx-1:02d}.jpg"
                    cv2.imwrite(str(preview_file), result.plot())
                del result
        cpu_confidences.append(scores)
        if len(scores) >= MAX_DET:
            nms_saturated += 1
        if idx % 25 == 0 or idx == len(entries):
            print(f"  Completed {idx}/{len(entries)} images")
    if nms_saturated:
        print(f"WARNING: {nms_saturated} image(s) reached max_det={MAX_DET}; count errors may be capped.")
    return cpu_confidences

print("\nVALIDATION COUNTING: one image at a time...")
val_confs = predict_cpu_confidences(val_rec, LOW_CONF)
val_gt = np.array([r["gt"] for r in val_rec], dtype=int)

rows = []
for threshold in CONF_GRID:
    predicted = np.array([int((c >= threshold).sum()) for c in val_confs], dtype=int)
    err = predicted - val_gt
    rows.append({
        "confidence": float(threshold),
        "count_MAE": float(np.mean(np.abs(err))),
        "count_RMSE": float(np.sqrt(np.mean(err.astype(float) ** 2))),
        "exact_count_accuracy": float(np.mean(predicted == val_gt)),
        "within_one_accuracy": float(np.mean(np.abs(err) <= 1)),
        "count_bias": float(np.mean(err)),
    })

tdf = pd.DataFrame(rows).sort_values(
    ["count_MAE", "exact_count_accuracy", "confidence"],
    ascending=[True, False, True],
)
tdf.to_csv(OUT / "metrics" / "validation_count_thresholds.csv", index=False)
COUNT_CONF = float(tdf.iloc[0]["confidence"])
print(f"VALIDATION-SELECTED COUNT CONFIDENCE: {COUNT_CONF:.2f}")
print("Best VAL counting:", tdf.iloc[0].to_dict())

plotdf = tdf.sort_values("confidence")
plt.figure(figsize=(8, 4))
plt.plot(plotdf["confidence"], plotdf["count_MAE"], marker="o")
plt.axvline(COUNT_CONF, ls="--", label=f"Selected confidence={COUNT_CONF:.2f}")
plt.xlabel("Detection confidence threshold")
plt.ylabel("Validation fish-count MAE")
plt.title("Counting threshold tuned ONLY on validation")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "count_threshold_validation.png", dpi=190)
plt.show()
plt.close()

del val_confs
tidy()

# ----------------------- LOCKED TEST EVALUATION ------------------
# Do not adjust this checkpoint/threshold using test feedback.
test_detection = None
test_counting = None

if RUN_FINAL_TEST:
    print("\nTEST DETECTION mAP (batch=1)...")
    test_metrics = best_model.val(
        data=str(DATA_YAML), split="test", imgsz=IMG_SIZE,
        device=DEVICE, batch=1, workers=0,
        conf=0.001, max_det=MAX_DET, plots=True,
        project=str(OUT / "evaluation"), name="test_memory_safe",
        exist_ok=True, verbose=False,
    )
    test_detection = summarize_metrics(test_metrics)
    test_detection["stage_speed_ms"] = {
        key: float(value)
        for key, value in getattr(test_metrics, "speed", {}).items()
        if isinstance(value, (int, float, np.number))
    }
    print("HELD-OUT INTERNAL TEST DETECTION:", test_detection)
    del test_metrics
    tidy()

    print("\nTEST COUNTING: one image at a time...")
    test_confs = predict_cpu_confidences(
        test_rec, COUNT_CONF, save_previews=RUN_PREDICTION_PREVIEW,
    )
    gt = np.array([r["gt"] for r in test_rec], dtype=int)
    counts_pred = np.array([int((c >= COUNT_CONF).sum()) for c in test_confs], dtype=int)
    delta = counts_pred - gt
    test_counting = {
        "count_MAE": float(np.mean(np.abs(delta))),
        "count_RMSE": float(np.sqrt(np.mean(delta.astype(float) ** 2))),
        "count_bias": float(np.mean(delta)),
        "exact_count_accuracy": float(np.mean(counts_pred == gt)),
        "within_one_accuracy": float(np.mean(np.abs(delta) <= 1)),
        "confidence_threshold": COUNT_CONF,
    }
    pd.DataFrame({
        "image": [r["image"].name for r in test_rec],
        "gt_fish_count": gt,
        "pred_fish_count": counts_pred,
        "count_error": delta,
    }).to_csv(OUT / "metrics" / "test_counting_per_image.csv", index=False)

    plt.figure(figsize=(6, 6))
    plt.scatter(gt, counts_pred, alpha=0.6)
    upper = int(max(gt.max(), counts_pred.max())) + 1
    plt.plot([0, upper], [0, upper], ls="--")
    plt.xlabel("Ground-truth annotated fish count")
    plt.ylabel("Predicted fish count")
    plt.title("Internal held-out test: image-level fish counts")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT / "figures" / "test_count_scatter.png", dpi=190)
    plt.show()
    plt.close()
    print("HELD-OUT INTERNAL TEST COUNTING:", test_counting)
    del test_confs
    tidy()

# -------------------------- SAVE OUTPUTS ------------------------
MODEL_WEIGHTS = str(globals().get("MODEL_WEIGHTS", "yolo26s.pt"))
DATASET_NAME = str(globals().get("DATASET_NAME", "Fish Defect Detection.v4i.coco"))
source_name = str(globals().get("SOURCE", DATA_YAML))
MAX_EPOCHS = int(globals().get("MAX_EPOCHS", 350))
EARLY_STOP_PATIENCE = int(globals().get("EARLY_STOP_PATIENCE", 45))

policy = {
    "weights_file": "model1_fish_detector_best.pt",
    "source_best_checkpoint": str(BEST),
    "architecture": MODEL_WEIGHTS,
    "imgsz": IMG_SIZE,
    "confidence_for_counting": COUNT_CONF,
    "nms_iou_for_counting": COUNT_IOU,
    "max_det": MAX_DET,
    "class_names": {"0": "Fish"},
    "source_dataset": DATASET_NAME,
    "split_type": "grouped internal holdout (verify capture-session independence)",
    "independent_external_test": False,
    "training": training_summary,
    "validation_detection": val_detection,
    "internal_heldout_test_detection": test_detection,
    "internal_heldout_test_counting": test_counting,
    "warning": "Do not use test metrics for further tuning. Fresh new-session evaluation is needed for external generalization claims.",
}

deploy_pt = OUT / "model1_fish_detector_best.pt"
if BEST.resolve() != deploy_pt.resolve():
    shutil.copy2(BEST, deploy_pt)
(OUT / "model1_deployment_policy.json").write_text(
    json.dumps(policy, indent=2), encoding="utf-8"
)

readme = (
    "TAMBAN MODEL 1 — POST-TRAINING EVALUATION\n"
    f"Dataset: {DATASET_NAME}\nSource: {source_name}\n"
    f"Model: {MODEL_WEIGHTS}\nCheckpoint: {BEST}\n"
    f"Validation images: {len(val_rec)}\nTest images: {len(test_rec)}\n"
    f"Epoch limit: {MAX_EPOCHS}; early-stop patience: {EARLY_STOP_PATIENCE}\n"
    f"COUNT_CONF (chosen from validation only): {COUNT_CONF:.2f}\n"
    "Outputs: metrics/, figures/, evaluation/, predictions/, model1_deployment_policy.json\n"
    "No retraining performed in this evaluation cell.\n"
    "Internal group-exclusive test metrics are not independent external accuracy.\n"
    "This detector learns whole-fish boxes, not anatomical width/cracks.\n"
)
(OUT / "README.txt").write_text(readme, encoding="utf-8")

print("\n" + "=" * 65)
print("POST-TRAINING EVALUATION COMPLETE — NO RETRAINING")
print("Results folder:", OUT)
print("Deployment checkpoint:", deploy_pt)
print("Validation metrics:", val_detection)
print("Selected count confidence:", COUNT_CONF)
print("Test detection metrics:", test_detection)
print("Test counting metrics:", test_counting)
print("=" * 65)


In [ ]:

from pathlib import Path
import torch
from ultralytics import YOLO

BASE = Path(
    r"C:\Users\User\Downloads\Tamban_Thesis_Main_Repository"
    r"\Thesis\sardinella-lemuru-ml\notebooks"
)

RUNS = (
    BASE / "Tamban_Model1_Retrain"
    / "model1_20261101T161439Z"
    / "runs"
)

# Your already-trained checkpoint
BEST = (
    RUNS / "model1_yolo26s"
    / "weights" / "best.pt"
)

# Same cleaned training/validation dataset
DATA = (
    BASE / "tamban_model1_1900_work"
    / "yolo_cleaned" / "data.yaml"
)

def main():
    assert torch.cuda.is_available(), "CUDA GPU unavailable"
    assert BEST.exists(), f"Missing checkpoint: {BEST}"
    assert DATA.exists(), f"Missing dataset: {DATA}"

    print("GPU:", torch.cuda.get_device_name(0))
    print("Starting from:", BEST)
    print("Stage 2 fine-tuning: up to 60 epochs")

    model = YOLO(str(BEST))

    model.train(
        data=str(DATA),

        # Additional training
        epochs=60,
        patience=20,

        # RTX 2050 - 4 GB VRAM
        imgsz=768,
        batch=1,
        device=0,
        amp=True,
        workers=2,
        cache=False,

        # Lower learning rate
        optimizer="AdamW",
        lr0=0.0002,
        lrf=0.1,
        cos_lr=True,
        warmup_epochs=2,

        # Moderate augmentation
        degrees=10,
        translate=0.06,
        scale=0.25,
        fliplr=0.5,
        flipud=0.0,
        hsv_h=0.01,
        hsv_s=0.25,
        hsv_v=0.25,
        mosaic=0.15,
        close_mosaic=10,

        # Checkpoints and reports
        max_det=132,
        save=True,
        save_period=10,
        plots=True,
        val=True,
        seed=42,

        # Separate output from original
        project=str(RUNS),
        name="model1_yolo26s_stage2",
        exist_ok=False,
        resume=False,
    )

    stage2_dir = Path(model.trainer.save_dir)
    print("\nSTAGE 2 TRAINING COMPLETE")
    print("Stage 2 checkpoint:",
          stage2_dir / "weights" / "best.pt")
    print("Original checkpoint:", BEST)
    print("Original checkpoint was not overwritten.")

if __name__ == "__main__":
    main()
